In [3]:
import duckdb as ddb
import glob
import os
import pandas as pd
import json

In [3]:
def sync_json_files(source_dir, target_dir):
    # 1. Get a set of all .json files in the source directory
    # Using a set allows for efficient comparison
    source_files = {f for f in os.listdir(source_dir) if f.endswith('.json')}
    
    # 2. Iterate through files in the target directory
    for filename in os.listdir(target_dir):
        # Only process .json files
        if filename not in source_files:
            file_path = os.path.join(target_dir, filename)
            try:
                os.remove(file_path)
            except Exception as e:
                print(f"Error deleting {filename}: {e}")
                continue

In [ ]:
factsDir = r'D:\EDGAR_Data_Analytics\Data\companyfacts'
submDir = r'D:\EDGAR_Data_Analytics\Data\submissions'
sync_json_files(factsDir, submDir)

In [ ]:
conn = ddb.connect('D:\\EDGAR_Data_Analytics\\Data\\secFilingsDB.db')
conn.execute("""CREATE OR REPLACE TABLE financialData (
    cik INTEGER,
    source VARCHAR,
    financialMetric VARCHAR,
    label VARCHAR,
    description VARCHAR,
    units VARCHAR,
    financialYear INTEGER,
    financialPeriod VARCHAR,
    endDate DATE,        -- Converted from string to DATE
    accn VARCHAR,
    value DOUBLE         -- Matches pandas float64
);
CHECKPOINT;""")

res = conn.execute("SELECT cik, facts FROM raw_data limit 3")

# 2. Iterate until all chunks are pulled
while True:
    df_chunk = res.fetch_df_chunk(1)
    # fetch_df_chunk() returns an empty DataFrame when finished
    if df_chunk.empty:
        break
    
    dfToLoad = pd.DataFrame({
            'cik': pd.Series(dtype='int'),
            'source': pd.Series(dtype='str'),
            'financialMetric': pd.Series(dtype='str'),
            'label': pd.Series(dtype='str'),
            'description': pd.Series(dtype='str'),
            'units': pd.Series(dtype='str'),
            'financialYear': pd.Series(dtype='int'),
            'financialPeriod': pd.Series(dtype='str'),
            'endDate': pd.Series(dtype='str'),
            'accn': pd.Series(dtype='str'),
            'value': pd.Series(dtype='float')
        })
    for i in df_chunk.index:
        try:
            cikNum = df_chunk.at[i, 'cik']
            jsn = json.loads(df_chunk.at[i, 'facts'])
            df_flat = pd.DataFrame()
            df_flat = pd.json_normalize(jsn,  max_level=1).T
            df_flat.reset_index(inplace=True)
            df_flat.columns = ['path', 'json']
            df_flat['units'] = df_flat['json'].apply(lambda x: list(x['units'].keys()))
            df_flat['label'] = df_flat['json'].apply(lambda x: x['label'])
            df_flat['description'] = df_flat['json'].apply(lambda x: x['description'])
            df_flat = df_flat.explode('units').reset_index(drop=True)
            df_flat['source'] = df_flat['path'].apply(lambda x: x.split('.')[0])
            df_flat['financialMetric'] = df_flat['path'].apply(lambda x: x.split('.')[1])
            df_flat.drop(columns=['path'], inplace=True)
            df_flat['records'] = df_flat.apply(lambda row: row['json']['units'][row['units']], axis=1)
            df_flat.drop(columns=['json'], inplace=True)
            df_flat = df_flat.explode('records').reset_index(drop=True)
            new_cols = pd.json_normalize(df_flat['records'])
            new_cols = new_cols.reindex(columns=['fy', 'fp', 'end', 'accn', 'val'])
            df_flat.drop(columns=['records'], inplace=True)
            df_flat = df_flat.join(new_cols)
            df_flat['cik'] = cikNum
            df_flat.rename(columns = {'fy':'financialYear', 'fp':'financialPeriod', 'end':'endDate', 'val':'value'}, inplace=True)
            dfToLoad = pd.concat([dfToLoad, df_flat], ignore_index=True)
        except:
            continue
    
    conn.execute("""
                INSERT INTO financialData 
                SELECT 
                    cik,
                    source,
                    financialMetric,
                    label,
                    description,
                    units,
                    financialYear,
                    financialPeriod,
                    CAST(endDate AS DATE) AS endDate,
                    accn,
                    value
                FROM dfToLoad;
                CHECKPOINT;
                """)
conn.execute("""
DROP TABLE raw_data;
CHECKPOINT;
""")
conn.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))